# 🏛️ MEF · Canon y sobrecanon petrolero recaudado por cada municipalidad (2005–hoy)

PERUPETRO publica el canon **por municipalidad** solo en 1993–2002 (Excel de Distribución) y el año en curso (Excel de Canon Municipalidades). Desde 2003 entrega el total municipal a un intermediario (CND hasta 2006; PCM desde 2007), que hace el reparto. Este notebook completa los años que faltan con el **MEF – Consulta Amigable de Ingresos** (`apps5.mineco.gob.pe/transparenciaingresos/mensual/`), monto **recaudado** por cada municipalidad.

**Cómo lee la consulta:** igual que los scrapers de `FISCAL/` (gasto): la ruta va **en la URL** (`Navegar_N.aspx?…&0=&10=<fuente>&11=<tipo>&…&y=AAAA`), con `requests`, sin navegador. La primera clave vacía es la que se lista. Claves conocidas por el `exportUrl` de la página: 10 fuente, 11 tipo de recurso, 2 sector; las demás (rubro, nivel de gobierno, departamento, municipalidad) **se descubren solas** probando claves y leyendo la cabecera de la tabla, y cada filtro se valida con el breadcrumb.

- **Desde 2007**: fuente 5 (Recursos determinados) → rubro 18 → tipos «5: Canon y sobrecanon petrolero» y «K: Sub cuenta – …» (se suman) → gobiernos locales → departamento → municipalidad.
- **2005–2006**: fuente 01 (Canon y sobrecanon) → tipo de recurso; el detalle municipal solo está en «0: NORMAL», que reúne **todos** los canones. Con `SI_NO_HAY_PETROLERO = "Usar NORMAL"` se toma ese tipo y queda rotulado «todos los canones» en el CSV y en el mapa.
- **2003–2004**: sin detalle por municipalidad (el canon petrolero municipal figura a nombre del Consejo Nacional de Descentralización).
- **Resultado:** `DATA/PP_canon_municipal_mef_anio.csv` y `CONTROL/CONTROL PP canon_municipal_mef.csv` (suma de municipalidades vs. total del departamento en el MEF, y vs. el total municipal de PERUPETRO: *recaudado* y *transferido* pueden diferir por fechas de registro).

**Bloque 1** importa las librerías. **Bloque 2** descarga y controla.

In [ ]:
#@title Bloque 1 · Librerías { display-mode: "form" }
import importlib
for m in ("requests", "bs4", "pandas"):
    importlib.import_module(m)
print("✅ Librerías listas")


In [ ]:
#@title Bloque 2 · Descargar y controlar { display-mode: "form" }
ANIO_INICIO = 2003  #@param {type:"integer"}
ANIO_FIN = 2026  #@param {type:"integer"}
SI_NO_HAY_PETROLERO = "Usar NORMAL (todos los canones)"  #@param ["Usar NORMAL (todos los canones)", "Dejar sin dato"]
DEPARTAMENTO = "16: LORETO"  #@param ["TODOS", "16: LORETO", "25: UCAYALI", "20: PIURA", "24: TUMBES", "10: HUANUCO", "08: CUSCO"]
import os, re, unicodedata, pandas as pd
from datetime import date
DRIVE_FOLDER_ID = "1dOOcsVyw915yKg6BzSdICVfCpYTiyeaq"            # DATA
DRIVE_FOLDER_CONTROL_ID = "1TKjS2Wjpnf26oladjhqstmN5c9Irq-Aj"    # CONTROL
FUENTE = "MEF, Consulta Amigable de Ingresos (recaudado)"
ruta_local = "/content/pp_mef"; os.makedirs(ruta_local, exist_ok=True)

# ── Google Drive ──
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)
def archivos_en(folder_id):
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false", fields="nextPageToken, files(id,name)",
                               pageSize=1000, pageToken=token, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out
def subir(ruta_archivo, folder_id, existentes, mime="text/csv"):
    nombre = os.path.basename(ruta_archivo)
    media = MediaFileUpload(ruta_archivo, mimetype=mime, resumable=True)
    if nombre in existentes:
        drive.files().update(fileId=existentes[nombre], media_body=media, supportsAllDrives=True).execute()
    else:
        existentes[nombre] = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                                  fields="id", supportsAllDrives=True).execute()["id"]
def bajar(nombre, existentes):
    if nombre not in existentes:
        return None
    p = os.path.join(ruta_local, nombre)
    with open(p, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive.files().get_media(fileId=existentes[nombre], supportsAllDrives=True)); hecho = False
        while not hecho:
            _, hecho = dl.next_chunk()
    return pd.read_csv(p, encoding="utf-8-sig")
datos_drive, control_drive = archivos_en(DRIVE_FOLDER_ID), archivos_en(DRIVE_FOLDER_CONTROL_ID)

# ── Lector por URL: canon y sobrecanon petrolero recaudado por cada municipalidad (MEF, Consulta Amigable de Ingresos) ──
# Igual que los scrapers de FISCAL/ (Consulta Amigable de gasto): la ruta va en la URL, con requests y sin navegador.
#   Navegar_N.aspx?_tgt=frame&_uhc=yes&0=&10=<fuente>&11=<tipo>&…&<clave a listar>=&y=AAAA
#   · cada clave numérica es una dimensión; la PRIMERA clave vacía es la que se lista (lo que viene después se ignora);
#   · claves vistas en el exportUrl de la página: 0 = TOTAL, 10 = Fuente de financiamiento, 11 = Tipo de recurso, 2 = Sector;
#   · las demás (rubro, nivel de gobierno, gob. local, departamento, municipalidad) se DESCUBREN: se prueba cada clave y se lee
#     qué dimensión muestra la cabecera de la tabla. Cada filtro se valida con el breadcrumb (como en FISCAL).
import re, time, random, unicodedata
import requests
from bs4 import BeautifulSoup

BASE = "https://apps5.mineco.gob.pe/transparenciaingresos/mensual"
PAGINAS = (3, 1, 2, 4, 5, 6, 7)                 # Navegar_N.aspx: el número cambia; se calibra
FORMATOS = [{"nombre": "navegador", "pre": {"_tgt": "frame", "_uhc": "yes"}, "post": {}},
            {"nombre": "simple", "pre": {}, "post": {}}]
CLAVES_CONOCIDAS = {"Fuente de Financiamiento": "10", "Tipo de Recurso": "11", "Sector": "2"}
CANDIDATAS = ["1", "37", "5", "7", "12", "13", "14", "15", "16", "17", "18", "19", "20", "3", "4", "6", "8", "9", "21", "22",
              "23", "24", "25", "26", "27", "28", "29", "30", "31", "32", "33", "34", "35", "36", "38", "39", "40"]
DIM_BUSCADAS = ("Rubro", "Nivel de Gobierno", "Gob.Loc./Mancom.", "Departamento", "Municipalidad")
_CODIGO = re.compile(r"^([\w\-\.]+)\s*:")

def norm(t):
    t = unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", t).strip().lower()

def a_numero(t):
    t = (t or "").replace(",", "").replace("\xa0", "").strip()
    try:
        return float(t)
    except ValueError:
        return None

class Pagina:
    """dim: dimensión listada (cabecera de la tabla) · filtros: breadcrumb [(etiqueta, recaudado)] ·
    filas: [(codigo, etiqueta, recaudado)] (recaudado exacto del valor del radio «código/PIA/PIM/recaudado»)."""
    def __init__(self, html, url=""):
        self.url = url
        s = BeautifulSoup(html, "html.parser")
        self.tiene_tabla = s.find("table", class_="Data") is not None
        mt = s.find("table", class_="MapTable")
        celdas = mt.find("tr").find_all("td") if mt and mt.find("tr") else []
        self.dim = " ".join(celdas[1].get_text(" ", strip=True).split()) if len(celdas) > 1 else ""
        hist = s.find("table", class_="History")
        self.filtros = []
        for tr in (hist.find_all("tr") if hist else []):
            tds = tr.find_all("td")
            if len(tds) >= 3:
                et = " ".join(tds[1].get_text(" ", strip=True).split())
                self.filtros.append((et, a_numero(tds[-1].get_text(strip=True))))
        self.filas = []
        data = s.find("table", class_="Data")
        for tr in (data.find_all("tr") if data else []):
            r = tr.find("input", attrs={"type": "radio"})
            tds = tr.find_all("td")
            if r is None or len(tds) < 3:
                continue
            et = " ".join(tds[1].get_text(" ", strip=True).split())
            partes = (r.get("value") or "").split("/")
            rec = a_numero(partes[-1]) if len(partes) >= 3 else None
            if rec is None:
                rec = a_numero(tds[-1].get_text(strip=True))
            m = _CODIGO.match(et)
            cod = partes[0] if partes and partes[0] else (m.group(1) if m else et)
            self.filas.append((cod.strip(), et, rec or 0.0))

class MEFIngresosURL:
    def __init__(self, log=print, pausa=(0.8, 1.6), base=BASE):
        self.log, self.pausa, self.base = log, pausa, base.rstrip("/")
        self.s = requests.Session()
        self.s.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                                             "Chrome/124.0 Safari/537.36", "Accept-Language": "es-PE,es;q=0.9",
                               "Referer": f"{self.base}/default.aspx"})
        self.pagina = self.formato = None
        self.claves = dict(CLAVES_CONOCIDAS)
        self._anio_sesion = None

    def _get(self, anio, ruta, listar, pagina=None, formato=None, reintentos=3):
        """ruta = [(clave, valor)…]; listar = clave a listar (vacía). Devuelve (Pagina | None, error)."""
        pagina, formato = pagina or self.pagina, formato or self.formato
        if self._anio_sesion != anio:                    # como el navegador: la página principal fija el año en la sesión
            try:
                self.s.get(f"{self.base}/default.aspx", params={"y": anio}, timeout=40)
            except requests.RequestException:
                pass
            self._anio_sesion = anio
        p = dict(formato["pre"]); p["0"] = ""
        for k, v in ruta:
            p[k] = v
        p[listar] = ""
        p["y"] = anio
        p.update(formato["post"])
        err = ""
        for i in range(reintentos):
            try:
                r = self.s.get(f"{self.base}/Navegar_{pagina}.aspx", params=p, timeout=40)
                if r.status_code == 200:
                    pg = Pagina(r.content, r.url)
                    if pg.tiene_tabla:
                        time.sleep(random.uniform(*self.pausa))
                        return pg, None
                    err = "SIN_TABLA"
                else:
                    err = f"HTTP {r.status_code}"
            except requests.RequestException as e:
                err = f"{type(e).__name__}: {e}"
            time.sleep(min(20, 2 ** i + random.random()))
        return None, err

    def calibrar(self, anio):
        """Elige el Navegar_N y el formato que listan las fuentes de financiamiento."""
        for n in PAGINAS:
            for f in FORMATOS:
                pg, err = self._get(anio, [], "10", pagina=n, formato=f, reintentos=1)
                ok = pg is not None and pg.filas and norm(pg.dim).startswith("fuente")
                self.log(f"   · Navegar_{n}.aspx ({f['nombre']}): {'OK, ' + str(len(pg.filas)) + ' fuentes' if ok else (err or 'otra tabla: ' + (pg.dim if pg else ''))}")
                if ok:
                    self.pagina, self.formato = n, f
                    return True
        return False

    def descubrir(self, anio, ruta, buscadas):
        """Prueba claves candidatas bajo `ruta` y anota cuál lista cada dimensión buscada (por la cabecera de la tabla)."""
        faltan = [d for d in buscadas if d not in self.claves]
        usadas = {k for k, _ in ruta} | set(self.claves.values())
        for k in CANDIDATAS:
            if not faltan:
                break
            if k in usadas:
                continue
            pg, _ = self._get(anio, ruta, k, reintentos=1)
            if pg is None or not pg.dim:
                continue
            d = next((x for x in faltan if norm(pg.dim).startswith(norm(x)[:10])), None)
            if d:
                self.claves[d] = k; faltan.remove(d)
                self.log(f"   · clave {k} = {d}")
        return faltan

    def listar(self, anio, ruta, dim):
        """Filas de la dimensión `dim` bajo `ruta`, validando que la tabla liste esa dimensión."""
        k = self.claves.get(dim)
        if k is None:
            return None, f"clave desconocida para «{dim}»"
        pg, err = self._get(anio, ruta, k)
        if err:
            return None, err
        if not norm(pg.dim).startswith(norm(dim)[:10]):
            return None, f"se pidió «{dim}» y la tabla lista «{pg.dim}»"
        return pg, None

    @staticmethod
    def elegir(pg, patron):
        return next(((c, e, v) for c, e, v in pg.filas if re.search(patron, e, re.I)), None)

    @staticmethod
    def filtro_aplicado(pg, etiqueta):
        """El breadcrumb termina en el filtro elegido (p. ej. «Departamento 16: LORETO»)."""
        return bool(pg.filtros) and norm(etiqueta) in norm(pg.filtros[-1][0])

    def ruta_tipos(self, anio):
        """Esquema del año: desde 2007 fuente 5 → rubro 18; hasta 2006 fuente 01 (Canon y sobrecanon). Devuelve (ruta, tipos)."""
        pg, err = self.listar(anio, [], "Fuente de Financiamiento")
        if err:
            return None, [], err
        for patron, con_rubro in ((r"^0?5:\s*RECURSOS DETERMINADOS", True), (r"^0?1:\s*CANON Y SOBRECANON", False)):
            f = self.elegir(pg, patron)
            if not f:
                continue
            ruta = [(self.claves["Fuente de Financiamiento"], f[0])]
            if con_rubro:
                if "Rubro" not in self.claves:
                    self.descubrir(anio, ruta, ["Rubro"])
                pr, err = self.listar(anio, ruta, "Rubro")
                if err:
                    return None, [], f"rubro: {err}"
                r18 = self.elegir(pr, r"^18:")
                if not r18:
                    return None, [], "sin rubro 18"
                ruta = ruta + [(self.claves["Rubro"], r18[0])]
            pt, err = self.listar(anio, ruta, "Tipo de Recurso")
            if err:
                return None, [], f"tipo de recurso: {err}"
            return ruta, pt.filas, None
        return None, [], "sin fuente 5 (Recursos determinados) ni 01 (Canon y sobrecanon)"

    def hasta_departamentos(self, anio, ruta, tipo):
        """Agrega tipo de recurso y nivel M (y gob. local M si existe) y lista los departamentos."""
        r = ruta + [(self.claves["Tipo de Recurso"], tipo[0])]
        if "Nivel de Gobierno" not in self.claves:
            self.descubrir(anio, r, ["Nivel de Gobierno"])
        pn, err = self.listar(anio, r, "Nivel de Gobierno")
        if err:
            return None, None, f"nivel de gobierno: {err}"
        m = self.elegir(pn, r"^M:")
        if not m:
            return None, None, "sin gobiernos locales (nivel M) en este tipo de recurso"
        r = r + [(self.claves["Nivel de Gobierno"], m[0])]
        if "Gob.Loc./Mancom." not in self.claves:          # solo en años recientes; en gasto es la clave 37
            pg, _ = self._get(anio, r, "37", reintentos=1)
            if pg is not None and norm(pg.dim).startswith("gob"):
                self.claves["Gob.Loc./Mancom."] = "37"; self.log("   · clave 37 = Gob.Loc./Mancom.")
        if "Departamento" not in self.claves:
            self.descubrir(anio, r, ["Departamento"])
        if "Gob.Loc./Mancom." in self.claves:
            pg, err = self.listar(anio, r, "Gob.Loc./Mancom.")
            if not err:
                g = self.elegir(pg, r"^M:")
                if g:
                    r = r + [(self.claves["Gob.Loc./Mancom."], g[0])]
        pd_, err = self.listar(anio, r, "Departamento")
        if err:
            return None, None, f"departamento: {err}"
        return r, pd_, None

    def municipalidades(self, anio, ruta_dep, dep):
        """Lista las municipalidades del departamento elegido; total = breadcrumb del departamento."""
        r = ruta_dep + [(self.claves["Departamento"], dep[0])]
        if "Municipalidad" not in self.claves:
            self.descubrir(anio, r, ["Municipalidad"])
        pm, err = self.listar(anio, r, "Municipalidad")
        if err:
            return None, None, f"municipalidad: {err}"
        if not self.filtro_aplicado(pm, dep[1].split(":", 1)[1] if ":" in dep[1] else dep[1]):
            return None, None, f"el filtro de departamento no se aplicó (breadcrumb: {pm.filtros[-1][0] if pm.filtros else '–'})"
        total = pm.filtros[-1][1] if pm.filtros else None
        return pm.filas, total, None

def nombre_municipalidad(etiqueta):
    """«01-301422: MUNICIPALIDAD PROVINCIAL DE MAYNAS - IQUITOS» → (código, nivel, municipalidad, provincia)."""
    cod, _, txt = etiqueta.partition(":")
    txt = txt.strip().upper()
    m = re.match(r"MUNICIPALIDAD\s+(PROVINCIAL|DISTRITAL)\s+(?:(?:DE\s+LA|DEL|DE)\s+)?(.*)$", txt)
    if not m:
        return cod.strip(), "", txt, ""
    nivel, nom = m.group(1), m.group(2).strip()
    if nivel == "PROVINCIAL":
        prov, _, capital = nom.partition(" - ")
        return cod.strip(), nivel, (capital or prov).strip(), prov.strip()
    return cod.strip(), nivel, nom, ""


def norm_may(t):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKD", str(t or "")).encode("ascii", "ignore").decode()).strip().upper()
TODOS = DEPARTAMENTO.strip().upper() == "TODOS"
# referencias de PERUPETRO (si existen en DATA): total municipal por departamento y año, y provincia de cada municipalidad
dist_pp = bajar("PP_canon_distribucion_mes.csv", datos_drive)
mun_pp = bajar("PP_canon_municipal_mes.csv", datos_drive)
pp_mun = (dist_pp[dist_pp["TIPO_BENEFICIARIO"] == "MUNICIPALIDADES"].assign(_D=dist_pp["DEPARTAMENTO"].map(norm_may))
          .groupby(["_D", "AÑO"])["MONTO_SOLES"].sum().to_dict() if dist_pp is not None else {})
prov_de = ({norm_may(m): norm_may(p) for m, p in zip(mun_pp["MUNICIPALIDAD"], mun_pp["PROVINCIA"]) if isinstance(p, str)}
           if mun_pp is not None else {})

m = MEFIngresosURL(log=print)
print("🔎 Calibrando la consulta…")
if not m.calibrar(min(ANIO_FIN, date.today().year)):
    raise SystemExit("❌ Ninguna variante de URL trajo la tabla de fuentes de financiamiento: el MEF cambió la página o bloqueó la consulta.")
filas, ctrl = [], []
for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    etq_dep = "TODOS" if TODOS else norm_may(DEPARTAMENTO.split(":", 1)[1])
    ruta, tipos, err = m.ruta_tipos(anio)
    if err:
        print(f"⚠️ {anio}: {err}"); ctrl.append({"AÑO": anio, "DEPARTAMENTO": etq_dep, "ESTADO": f"ERROR: {err}"}); continue
    resultados = []                                     # [(tipo, ruta hasta departamento, página de departamentos)]
    motivos = []
    for t in [t for t in tipos if re.search("PETROLER", t[1], re.I)]:
        r, pdep, e = m.hasta_departamentos(anio, ruta, t)
        if r: resultados.append((t, r, pdep))
        else: motivos.append(f"{t[1]}: {e}")
    usa_normal = False
    if not resultados and SI_NO_HAY_PETROLERO.startswith("Usar") and anio < 2007:
        t0 = next((t for t in tipos if re.match(r"^0:\s*NORMAL", t[1], re.I)), None)
        if t0:
            r, pdep, e = m.hasta_departamentos(anio, ruta, t0)
            if r: resultados, usa_normal = [(t0, r, pdep)], True
            else: motivos.append(f"{t0[1]}: {e}")
    if not resultados:
        msg = "SIN DETALLE MUNICIPAL (" + "; ".join(motivos or ["sin tipo de recurso petrolero"]) + ")"
        print(f"ℹ️ {anio}: {msg}"); ctrl.append({"AÑO": anio, "DEPARTAMENTO": etq_dep, "ESTADO": msg}); continue
    if TODOS:
        deps = sorted({(c, e) for _, _, pdep in resultados for c, e, v in pdep.filas if (v or 0) > 0})
    else:
        cod = DEPARTAMENTO.split(":")[0].strip()
        deps = [(cod, DEPARTAMENTO)]
    for cod_d, etq in deps:
        DEP = norm_may(etq.split(":", 1)[1] if ":" in etq else etq)
        suma = tot = 0.0; n = 0; detalle = []
        for t, r, pdep in resultados:
            dep = next(((c, e, v) for c, e, v in pdep.filas if c.lstrip("0") == cod_d.lstrip("0")), None)
            if dep is None:
                detalle.append(f"{t[1]}: sin {DEP}"); continue
            fs, total, e = m.municipalidades(anio, r, dep)
            if fs is None:
                detalle.append(f"{t[1]}: {e}"); continue
            for c, et, v in fs:
                if not re.search("MUNICIPALIDAD", et, re.I):
                    continue
                codigo, nivel, mu, prov = nombre_municipalidad(et)
                filas.append({"AÑO": anio, "DEPARTAMENTO": DEP, "CODIGO": codigo, "MUNICIPALIDAD_MEF": et.split(":", 1)[1].strip(),
                              "MUNICIPALIDAD": mu, "NIVEL": nivel, "PROVINCIA": prov or prov_de.get(norm_may(mu), ""),
                              "TIPO_RECURSO": t[1] + (" (todos los canones)" if usa_normal else ""), "RECAUDADO_SOLES": round(v, 2),
                              "FUENTE": FUENTE + (" · tipo NORMAL: todos los canones, no solo el petrolero" if usa_normal else "")})
                suma += v; n += 1
            tot += total or 0.0
            detalle.append(f"{t[1]}: S/ {total or 0:,.0f}")
        ref = pp_mun.get((DEP, anio))
        ctrl.append({"AÑO": anio, "DEPARTAMENTO": DEP, "TIPOS": " | ".join(detalle), "MUNICIPALIDADES": n,
                     "SUMA_MUNICIPALIDADES": round(suma, 2), "TOTAL_DEPARTAMENTO_MEF": round(tot, 2), "DIF": round(suma - tot, 2),
                     "PERUPETRO_MUNICIPAL": round(ref, 2) if ref else None, "MEF_SOBRE_PERUPETRO": round(suma / ref, 4) if ref else None,
                     "ESTADO": ("OK" if n and abs(suma - tot) <= max(1, n) else "REVISAR" if n else "SIN MUNICIPALIDADES")})
        print(f"{'✅' if n else 'ℹ️'} {anio} · {DEP}: {n} municipalidades · S/ {suma/1e6:,.2f} millones"
              + (" (todos los canones)" if usa_normal else "") + (f" · PERUPETRO S/ {ref/1e6:,.2f} millones" if ref else ""))
print("Claves usadas:", m.claves, "· página Navegar_", m.pagina)

# ── Guardar (se acumula: cada corrida reemplaza los departamentos y años que vuelve a leer) ──
nuevo = pd.DataFrame(filas)
previo = bajar("PP_canon_municipal_mef_anio.csv", datos_drive)
if previo is not None and len(nuevo):
    leidos = set(zip(nuevo["DEPARTAMENTO"], nuevo["AÑO"]))
    previo = previo[[(d, a) not in leidos for d, a in zip(previo["DEPARTAMENTO"], previo["AÑO"])]]
    nuevo = pd.concat([previo, nuevo], ignore_index=True)
elif previo is not None:
    nuevo = previo
if len(nuevo):
    nuevo = nuevo.sort_values(["DEPARTAMENTO", "AÑO", "NIVEL", "MUNICIPALIDAD", "TIPO_RECURSO"], ascending=[True, True, False, True, True])
    p = os.path.join(ruta_local, "PP_canon_municipal_mef_anio.csv"); nuevo.to_csv(p, index=False, encoding="utf-8-sig")
    subir(p, DRIVE_FOLDER_ID, datos_drive)
c = pd.DataFrame(ctrl); c.insert(0, "FECHA_EXTRACCION", date.today().strftime("%d/%m/%Y"))
previo_c = bajar("CONTROL PP canon_municipal_mef.csv", control_drive)
if previo_c is not None:
    leidos_c = set(zip(c["DEPARTAMENTO"], c["AÑO"]))
    c = pd.concat([previo_c[[(d, a) not in leidos_c for d, a in zip(previo_c["DEPARTAMENTO"], previo_c["AÑO"])]], c],
                  ignore_index=True).sort_values(["AÑO", "DEPARTAMENTO"])
p = os.path.join(ruta_local, "CONTROL PP canon_municipal_mef.csv"); c.to_csv(p, index=False, encoding="utf-8-sig")
subir(p, DRIVE_FOLDER_CONTROL_ID, control_drive)
print(f"\n☁️ DATA/PP_canon_municipal_mef_anio.csv ({len(nuevo)} filas) · CONTROL/CONTROL PP canon_municipal_mef.csv")
display(c[c["AÑO"].between(ANIO_INICIO, ANIO_FIN)].drop(columns=["FECHA_EXTRACCION"]))
